In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

Instalação das bibliotecas

In [ ]:
%pip install "psycopg[binary]" pandas pyarrow tqdm

Criação de arquivos para construção da base do banco

In [ ]:
import os

import pandas as pd
from tqdm.auto import tqdm


input_path = (
    "/content/drive/MyDrive/TCC_Chatbot/datasets/curated/banco_python_finalizado.parquet"
)

output_dir = (
    "/content/drive/MyDrive/TCC_Chatbot/datasets/postgresql/"
)

csv_path = os.path.join(
    output_dir,
    "answers.csv"
)

schema_path = os.path.join(
    output_dir,
    "schema.sql"
)


print("=" * 80)
print("PREPARAÇÃO DA BASE PARA POSTGRESQL")
print("=" * 80)


print("\nCriando diretório de saída...")

os.makedirs(
    output_dir,
    exist_ok=True
)


print("\nCarregando banco definitivo...")

df = pd.read_parquet(input_path)

df = df.reset_index(drop=True)

print(
    f"Total de registros: {len(df):,}"
)


required_columns = [
    "question",
    "answer",
    "code",
    "source",
    "language"
]


missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]


if missing_columns:

    raise ValueError(
        f"Colunas ausentes no banco: {missing_columns}"
    )


print("\nCriando IDs...")

df.insert(
    0,
    "id",
    range(len(df))
)


print("\nSelecionando colunas definitivas...")

df = df[
    [
        "id",
        "question",
        "answer",
        "code",
        "source",
        "language"
    ]
]


print("\nExportando answers.csv...")

df.to_csv(
    csv_path,
    index=False,
    encoding="utf-8",
    na_rep=""
)


print(
    f"Arquivo CSV criado: {csv_path}"
)


print("\nCriando schema.sql...")

schema = """\
CREATE TABLE IF NOT EXISTS answers (

    id INTEGER PRIMARY KEY,
    question TEXT NOT NULL,
    answer TEXT NOT NULL,
    code TEXT,
    source TEXT,
    language TEXT

);

CREATE INDEX IF NOT EXISTS idx_answers_source
ON answers(source);

CREATE INDEX IF NOT EXISTS idx_answers_question
ON answers(question);

\\copy answers
(
    id,
    question,
    answer,
    code,
    source,
    language
)
FROM 'answers.csv'
WITH (
    FORMAT csv,
    HEADER true,
    ENCODING 'UTF8'
);
"""


with open(
    schema_path,
    "w",
    encoding="utf-8"
) as schema_file:

    schema_file.write(schema)


print(
    f"Arquivo SQL criado: {schema_path}"
)


print("\n" + "=" * 80)
print("VALIDAÇÃO DA EXPORTAÇÃO")
print("=" * 80)


print(
    f"Registros no Parquet: {len(df):,}"
)


df_csv = pd.read_csv(
    csv_path,
    encoding="utf-8"
)


print(
    f"Registros no CSV: {len(df_csv):,}"
)


if len(df_csv) != len(df):

    raise ValueError(
        "A quantidade de registros no CSV é diferente "
        "da quantidade de registros no DataFrame."
    )


print("\nQuantidade de registros: OK")


print("\nPrimeiro registro:")

print(
    f"ID:       {df.iloc[0]['id']}"
)

print(
    f"Question: {df.iloc[0]['question']}"
)


print("\nÚltimo registro:")

print(
    f"ID:       {df.iloc[-1]['id']}"
)

print(
    f"Question: {df.iloc[-1]['question']}"
)


print("\n" + "=" * 80)
print("DISTRIBUIÇÃO POR SOURCE")
print("=" * 80)


source_distribution = (
    df.groupby("source")
    .size()
    .sort_values(ascending=False)
)


for source, count in source_distribution.items():

    print(
        f"{str(source):<25} "
        f"{count:,}"
    )


print("\n" + "=" * 80)
print("EXPORTAÇÃO CONCLUÍDA")
print("=" * 80)


print(
    f"\nCSV:    {csv_path}"
)

print(
    f"Schema: {schema_path}"
)

print(
    f"\nTotal exportado: {len(df):,}"
)